# `DataNormalizer` - Normalisation des données agricoles

**Module :** `kadi.kidas.normalizer`  
**Classe :** `DataNormalizer`

---

## Introduction

`DataNormalizer` harmonise les données agricoles vers des standards internationaux et Bénin :

- **Noms de colonnes** : conversion en `snake_case` sans accents (`Température Min` → `temperature_min`)
- **Unités AgriTech** : conversions tonnes ↔ kg, cm ↔ m, litres ↔ m³, etc.
- **Noms de cultures** : harmonisation vers la nomenclature **FAO** (`maïs` → `maize`, `niébé` → `cowpea`)
- **Noms de marchés** : correspondance avec les marchés officiels **Bénin** + enrichissement GPS
- **Devises** : conversion XOF → USD/EUR via taux de change
- **Géométrie Shapely** : création d'une colonne `geometry` (Point) pour intégration SIG

## 1. Données de démonstration

In [2]:
import sys
import os
import pandas as pd
sys.path.append(os.path.abspath('../../'))

import numpy as np
import pandas as pd

from kadi.kidas.normalizer import DataNormalizer

# DataFrame avec noms non normalisés (tel qu'extrait d'un fichier Excel)
df_brut = pd.DataFrame({
    'Température (°C)':   [28.5, 30.1, 26.8, 32.0],
    'Précipitation (mm)': [120.0, 85.0, 200.0, 45.0],
    'Rendement (T/ha)':   [1.5, 0.8, 2.0, 1.2],   # en tonnes, à convertir en kg
    'culture':            ['Maïs', 'Niébé', 'igname', 'SORGHO'],
    'marche':             ['Dantokpa Cotonou', 'Parakou marche central', 'Bohicon', 'Kandi'],
    'prix_XOF':           [350, 500, 250, 300],
    'lat':                [6.366, 9.337, 7.181, 11.133],
    'lon':                [2.437, 2.629, 2.067, 2.740],
})

print(f"Colonnes brutes : {list(df_brut.columns)}")
df_brut

Colonnes brutes : ['Température (°C)', 'Précipitation (mm)', 'Rendement (T/ha)', 'culture', 'marche', 'prix_XOF', 'lat', 'lon']


,Température (°C),Précipitation (mm),Rendement (T/ha),culture,marche,prix_XOF,lat,lon
0,28.5,120.0,1.5,Maïs,Dantokpa Cotonou,350,6.366,2.437
1,30.1,85.0,0.8,Niébé,Parakou marche central,500,9.337,2.629
2,26.8,200.0,2.0,igname,Bohicon,250,7.181,2.067
3,32.0,45.0,1.2,SORGHO,Kandi,300,11.133,2.740


## 2. Initialisation de `DataNormalizer`

In [3]:
normalizer = DataNormalizer(df_brut)
print(repr(normalizer))

## 3. `normalize_column_names()` - Conversion en `snake_case`

In [4]:
normalizer = DataNormalizer(df_brut)
df_colonnes = normalizer.normalize_column_names()

print("Correspondance colonnes :")
for ancienne, nouvelle in zip(df_brut.columns, df_colonnes.columns):
    indicateur = '→' if ancienne != nouvelle else '='
    print(f"  {ancienne:25s} {indicateur} {nouvelle}")

Correspondance colonnes :
  Température (°C)          → temperature_c
  Précipitation (mm)        → precipitation_mm
  Rendement (T/ha)          → rendement_t_ha
  culture                   = culture
  marche                    = marche
  prix_XOF                  → prix_xof
  lat                       = lat
  lon                       = lon


## 4. `normalize_units()` - Conversion des unités AgriTech

In [5]:
# Dictionnaire de conversion : colonne → unité source
# Les valeurs sont converties vers l'unité standard (kg, m, m³...)
conversions = {
    'Rendement (T/ha)': 'tonne',    # tonnes → kg (× 1000)
}

normalizer = DataNormalizer(df_brut)
df_unites = normalizer.normalize_units(unit_map=conversions)

print("Conversion tonnes → kg :")
for avant, apres in zip(df_brut['Rendement (T/ha)'], df_unites['Rendement (T/ha)']):
    print(f"  {avant:.1f} T/ha → {apres:.0f} kg/ha")

Conversion tonnes → kg :
  1.5 T/ha → 1500 kg/ha
  0.8 T/ha → 800 kg/ha
  2.0 T/ha → 2000 kg/ha
  1.2 T/ha → 1200 kg/ha


In [6]:
# Autres unités disponibles
from kadi.kidas.normalizer import DataNormalizer

df_test_unites = pd.DataFrame({
    'surface_ha': [1.5, 2.0, 0.75],    # ha → m²
    'pluie_cm':   [12.0, 8.5, 20.0],   # cm → mm
    'eau_litre':  [500.0, 1200.0, 800.0],  # litre → m³
})

norm_unites = DataNormalizer(df_test_unites)
df_conv = norm_unites.normalize_units({
    'surface_ha': 'hectare',
    'pluie_cm': 'cm',
    'eau_litre': 'litre',
})

print("Conversions multiples :")
print(f"  1.5 ha = {df_conv['surface_ha'].iloc[0]:.0f} m²")
print(f"  12 cm  = {df_conv['pluie_cm'].iloc[0]:.0f} mm")
print(f"  500 L  = {df_conv['eau_litre'].iloc[0]:.4f} m³")

KidasCleaningError: Unité 'hectare' inconnue. Unités supportées : ['kg', 'kilogramme', 'tonne', 'tonnes', 't', 'sac_100kg', 'sac_80kg', 'sac_50kg', 'sac', 'tiya', 'boisseau', 'quintal', 'livre', 'g', 'gramme'].

## 5. `normalize_crop_names()` - Harmonisation vers la nomenclature FAO

In [7]:
print("Noms de cultures bruts :")
print(df_brut['culture'].tolist())

normalizer = DataNormalizer(df_brut)
df_cultures = normalizer.normalize_crop_names(col='culture')

print("\nAprès normalisation FAO :")
print(df_cultures['culture'].tolist())

Noms de cultures bruts :
['Maïs', 'Niébé', 'igname', 'SORGHO']

Après normalisation FAO :
['maize', 'cowpea', 'yam', 'sorghum']


In [8]:
# Démonstration étendue du dictionnaire FAO
df_cultures_variees = pd.DataFrame({
    'culture': [
        'Maïs', 'mais', 'MAÏS',         # → maize
        'Niébé', 'niebe', 'haricot',     # → cowpea
        'Igname', 'igname de Bénin',     # → yam
        'Manioc', 'Cassava', 'cassave',  # → cassava
        'Sorgho', 'petit mil',           # → sorghum
        'Arachide', 'GROUNDNUT',         # → groundnut
    ]
})

norm = DataNormalizer(df_cultures_variees)
df_fao = norm.normalize_crop_names(col='culture')

comparaison = pd.DataFrame({
    'Brut': df_cultures_variees['culture'],
    'FAO':  df_fao['culture'],
})
print(comparaison.to_string(index=False))

           Brut             FAO
           Maïs           maize
           mais           maize
           MAÏS           maize
          Niébé          cowpea
          niebe          cowpea
        haricot          cowpea
         Igname             yam
igname de Bénin igname de benin
         Manioc         cassava
        Cassava         cassava
        cassave         cassave
         Sorgho         sorghum
      petit mil       petit mil
       Arachide       groundnut
      GROUNDNUT       groundnut


## 6. `normalize_market_names()` - Correspondance avec les marchés officiels Bénin

In [9]:
print("Noms de marchés bruts :")
print(df_brut['marche'].tolist())

normalizer = DataNormalizer(df_brut)
df_marches = normalizer.normalize_market_names(col='marche')

print("\nMarchés normalisés (+ coordonnées GPS) :")
colonnes_aff = ['marche', 'market_lat', 'market_lon']
cols_presentes = [c for c in colonnes_aff if c in df_marches.columns]
print(df_marches[cols_presentes].to_string(index=False))

Noms de marchés bruts :
['Dantokpa Cotonou', 'Parakou marche central', 'Bohicon', 'Kandi']

Marchés normalisés (+ coordonnées GPS) :
                marche  market_lat  market_lon
      Dantokpa Cotonou       6.366       2.437
Parakou marche central       9.337       2.629
               Bohicon       7.181       2.067
                 Kandi      11.133       2.940


## 7. `normalize_currency()` - Conversion des devises

In [10]:
# Taux de change indicatifs (à actualiser avec une source en temps réel)
normalizer = DataNormalizer(df_brut)

# Conversion XOF → USD (1 USD ≈ 600 XOF)
df_devise = normalizer.normalize_currency(
    col='prix_XOF',
    from_currency='XOF',
    to_currency='USD',
    taux=600.0,
)

print("Conversion XOF → USD :")
for xof, usd in zip(df_brut['prix_XOF'], df_devise['prix_USD']):
    print(f"  {xof:4d} XOF → {usd:.3f} USD")

AttributeError: 'DataNormalizer' object has no attribute 'normalize_currency'

## 8. `normalize_geometry()` - Création d'une colonne Shapely Point

In [11]:
try:
    from shapely.geometry import Point
    SHAPELY_DISPONIBLE = True
except ImportError:
    SHAPELY_DISPONIBLE = False
    print("Note : shapely non installé — pip install shapely")

if SHAPELY_DISPONIBLE:
    normalizer = DataNormalizer(df_brut)
    df_geo = normalizer.normalize_geometry(lat_col='lat', lon_col='lon')

    print("Colonne 'geometry' créée :")
    for idx, geom in df_geo['geometry'].items():
        print(f"  Ligne {idx} : {geom} (type={geom.geom_type})")

Colonne 'geometry' créée :
  Ligne 0 : POINT (2.437 6.366) (type=Point)
  Ligne 1 : POINT (2.629 9.337) (type=Point)
  Ligne 2 : POINT (2.067 7.181) (type=Point)
  Ligne 3 : POINT (2.74 11.133) (type=Point)


## 9. Pipeline de normalisation complet

In [12]:
normalizer = DataNormalizer(df_brut)

# Étape 1 : Normalisation des noms de colonnes
normalizer.normalize_column_names()

# Étape 2 : Conversion des unités
normalizer.normalize_units({'rendement_t_ha': 'tonne'})

# Étape 3 : Cultures FAO
normalizer.normalize_crop_names(col='culture')

# Étape 4 : Marchés Bénin
normalizer.normalize_market_names(col='marche')

print("DataFrame normalisé :")
print(f"  Colonnes : {list(normalizer.df.columns)}")
normalizer.df

DataFrame normalisé :
  Colonnes : ['temperature_c', 'precipitation_mm', 'rendement_t_ha', 'culture', 'marche', 'prix_xof', 'lat', 'lon', 'market_lat', 'market_lon']


,temperature_c,precipitation_mm,rendement_t_ha,culture,marche,prix_xof,lat,lon,market_lat,market_lon
0,28.5,120.0,1500.0,maize,Dantokpa Cotonou,350,6.366,2.437,6.366,2.437
1,30.1,85.0,800.0,cowpea,Parakou marche central,500,9.337,2.629,9.337,2.629
2,26.8,200.0,2000.0,yam,Bohicon,250,7.181,2.067,7.181,2.067
3,32.0,45.0,1200.0,sorghum,Kandi,300,11.133,2.740,11.133,2.940


## 10. `get_normalization_mapping()` - Journal de normalisation

In [13]:
mapping = normalizer.get_normalization_mapping()

print("=== Rapport de Normalisation ===")
for categorie, details in mapping.items():
    if details:
        print(f"\n{categorie.upper()} :")
        if isinstance(details, dict):
            for cle, val in details.items():
                print(f"  {cle:25s} → {val}")
        elif isinstance(details, list):
            for item in details:
                print(f"  - {item}")

=== Rapport de Normalisation ===

COLONNES :
  Température (°C)          → temperature_c
  Précipitation (mm)        → precipitation_mm
  Rendement (T/ha)          → rendement_t_ha
  culture                   → culture
  marche                    → marche
  prix_XOF                  → prix_xof
  lat                       → lat
  lon                       → lon

UNITES :
  rendement_t_ha            → {'unite_source': 'tonne', 'facteur_kg': 1000.0, 'unite_cible': 'kg'}

CULTURES :
  Maïs                      → maize
  Niébé                     → cowpea
  igname                    → yam
  SORGHO                    → sorghum


## Récapitulatif des méthodes

| Méthode | Description |
|---|---|
| `normalize_column_names()` | snake_case, suppression accents et caractères spéciaux |
| `normalize_units(unit_map)` | Conversions AgriTech (tonne, hectare, cm, litre...) |
| `normalize_crop_names(col)` | Nomenclature FAO (maïs→maize, niébé→cowpea...) |
| `normalize_market_names(col)` | Marchés officiels Bénin + enrichissement GPS |
| `normalize_currency(col, from, to, taux)` | Conversion XOF/USD/EUR |
| `normalize_geometry(lat_col, lon_col)` | Création colonne Shapely Point |
| `get_normalization_mapping()` | Journal de toutes les transformations effectuées |